# multi_4h_jma_atr_015

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (177).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `4h` |
| Code cells | 11 |
| Detected functions | resample_1hr_custom, __init__, compute_atr, compute_jma_and_signals, compute_atr_max, prepare_data, py_backtest_trades, calc_pnl, backtest_trades, __init__, compute_atr, compute_jma_and_signals |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** JMA + ATR crossover (detected)
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Binance_Crypto_2020_25.csv')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)
data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2020-01-01'
end_date = '2025-12-31'

# Define the tickers you want to filter
selected_tickers = ['BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'XRPUSDT', 'SOLUSDT']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('4h').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last'
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close']]

data = data_1hr
data

In [ ]:
data.dropna()
data

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from typing import List, Tuple, Optional

class ATRIndicator:
    def __init__(self, high_col='high', low_col='low', close_col='close'):
        self.high_col = high_col
        self.low_col = low_col
        self.close_col = close_col

    def compute_atr(self, df, period=14, shift=1):
        high, low, close = df[self.high_col], df[self.low_col], df[self.close_col]
        tr = pd.concat([
            high - low,
            (high - close.shift(1)).abs(),
            (low - close.shift(1)).abs()
        ], axis=1).max(axis=1)
        atr = tr.rolling(window=period, min_periods=1).mean()
        return atr.shift(shift)

def compute_jma_and_signals(df, smooth_length=14, normalization_period=72,
                                upper_threshold=1.75, lower_threshold=-0.75,
                                rs_period=55, rs_percentile=70):
    df = df.copy()
    price = df['close']

    # JMA calculation
    alpha = 2.0 / (smooth_length + 1)
    jma_values = [price.iloc[0]]
    for p in price.iloc[1:]:
        jma_values.append(jma_values[-1] + (p - jma_values[-1]) * alpha)
    df['JMA'] = jma_values

    # Normalization using shifted JMA to avoid look-ahead
    df['JMA_shifted'] = df['JMA'].shift(1)
    df['Normalized_JMA'] = (df['JMA_shifted'] - df['JMA_shifted'].rolling(normalization_period).mean()) / \
                          df['JMA_shifted'].rolling(normalization_period).std()

    # RS Volatility using previous bar's OHLC
    o = df['open'].shift(1).replace(0, np.nan)
    h = df['high'].shift(1)
    l = df['low'].shift(1)
    c = df['close'].shift(1).replace(0, np.nan)

    log_hc, log_ho = np.log(h / c), np.log(h / o)
    log_lc, log_lo = np.log(l / c), np.log(l / o)

    df['RS_Volatility'] = ((log_hc * log_ho + log_lc * log_lo).rolling(window=rs_period).mean()) * 1e6
    df['RS_Threshold'] = df['RS_Volatility'].rolling(normalization_period).quantile(rs_percentile / 100.0)

    # Signals based on shifted/lagged data
    df['Signal'] = 0
    df.loc[(df['Normalized_JMA'] > upper_threshold) & (df['RS_Volatility'] > df['RS_Threshold']), 'Signal'] = 1
    df.loc[(df['Normalized_JMA'] < lower_threshold) & (df['RS_Volatility'] > df['RS_Threshold']), 'Signal'] = 2

    return df

def compute_atr_max(group, multiplier=0.1):
    return pd.concat([
        group['ATR_14'].rolling(20).max() * multiplier,
        group['ATR_5'].rolling(20).max() * multiplier
    ], axis=1).max(axis=1)

def prepare_data(data):
    data = data.copy()
    data['Date'] = pd.to_datetime(data['Date'])

    # Ensure column names are lowercase for consistency
    data.columns = data.columns.str.lower()

    jma_results = [compute_jma_and_signals(group) for ticker, group in data.groupby('ticker')]
    data = pd.concat(jma_results, ignore_index=True)

    atr_calc = ATRIndicator()

    for period in [14, 5]:
        col_name = f'ATR_{period}'
        atr_results = [atr_calc.compute_atr(group, period=period, shift=1) for ticker, group in data.groupby('ticker')]
        data[col_name] = pd.concat(atr_results, ignore_index=True)

    atr_max_results = [compute_atr_max(group) for ticker, group in data.groupby('ticker')]
    data['ATR_max'] = pd.concat(atr_max_results, ignore_index=True)

    data['Signal'] = data.groupby('ticker', group_keys=False)['Signal'].shift(1)
    data = data.dropna(subset=['Signal'])
    data['Signal'] = data['Signal'].astype(np.int8)

    return data

def py_backtest_trades(
    data,
    ticker: str,
    capital: float,
    fee_rate: float,
    slippage_rate: float,
    entry_sl_pct: float
) -> List[Tuple]:
    """
    Backtest trades with entry SL and trailing stop-loss logic.

    Parameters
    ----------
    data : np.ndarray
        Array with columns [time, open, high, low, close, signal, atr_max].
    ticker : str
        Ticker symbol.
    capital : float
        Initial capital allocated per trade.
    fee_rate : float
        Fee rate applied per trade (as a fraction).
    slippage_rate : float
        Slippage applied to entry (as a fraction).
    entry_sl_pct : float
        Stop-loss percentage for entry candle.

    Returns
    -------
    trades : List[Tuple]
        Each trade tuple contains:
        (ticker, entry_time, exit_time, entry_price, exit_price,
         net_pnl, exit_reason, cumulative_pnl, direction)
    """

    def calc_pnl(entry_price: float, exit_price: float,
                 qty: float, direction: int) -> float:
        """Calculate net PnL after fees."""
        pnl = (exit_price - entry_price) * qty * direction
        fee = fee_rate * (entry_price + exit_price) * qty
        return pnl - fee

    n = data.shape[0]
    trades = []
    cumulative_pnl = 0.0

    # Position state
    position_open = False
    entry_price = sl_price = quantity = 0.0
    entry_time: Optional[pd.Timestamp] = None
    last_exit_time: Optional[pd.Timestamp] = None
    direction = 0

    # Previous candle values
    prev_high = prev_low = prev_close = prev_atr_max = 0.0

    for i in range(n):
        time = pd.to_datetime(data[i, 0])
        open_ = float(data[i, 1])
        high = float(data[i, 2])
        low = float(data[i, 3])
        close = float(data[i, 4])
        signal = int(data[i, 5])
        atr_max = float(data[i, 6])

        # ------------------------
        # ENTRY LOGIC
        # ------------------------
        if not position_open and time != last_exit_time and signal in (1, 2):
            direction = 1 if signal == 1 else -1
            entry_price = open_ * (1 + slippage_rate if direction == 1 else 1 - slippage_rate)
            quantity = capital / entry_price
            entry_time, position_open = time, True

            # Initial SL
            sl_price = entry_price * (1 - entry_sl_pct if direction == 1 else 1 + entry_sl_pct)

            # Check if entry SL is hit on the same candle
            if (direction == 1 and low <= sl_price) or (direction == -1 and high >= sl_price):
                exit_price = sl_price
                net_pnl = calc_pnl(entry_price, exit_price, quantity, direction)
                cumulative_pnl += net_pnl

                trades.append((
                    ticker, entry_time, time, entry_price, exit_price,
                    net_pnl, "Entry SL", cumulative_pnl,
                    "long" if direction == 1 else "short"
                ))

                position_open, last_exit_time = False, time
            continue

        # ------------------------
        # EXIT LOGIC
        # ------------------------
        if position_open and time != entry_time:
            exit_triggered = False
            exit_price = None
            exit_reason = None

            # Update trailing stop based on previous candle
            if i > 0:
                if direction == 1:  # Long
                    sl_price = max(sl_price, prev_close - prev_atr_max)
                else:  # Short
                    sl_price = min(sl_price, prev_close + prev_atr_max)

            # Check trailing SL
            if (direction == 1 and low <= sl_price) or (direction == -1 and high >= sl_price):
                exit_price = sl_price
                exit_reason = "Trailing SL"
                exit_triggered = True

            # Execute exit if triggered
            if exit_triggered:
                net_pnl = calc_pnl(entry_price, exit_price, quantity, direction)
                cumulative_pnl += net_pnl

                trades.append((
                    ticker, entry_time, time, entry_price, exit_price,
                    net_pnl, exit_reason, cumulative_pnl,
                    "long" if direction == 1 else "short"
                ))

                position_open, last_exit_time = False, time

        # ------------------------
        # UPDATE PREVIOUS CANDLE
        # ------------------------
        prev_high, prev_low, prev_close, prev_atr_max = high, low, close, atr_max

    return trades

def backtest_trades(data, initial_capital, fee_rate, slippage_rate, entry_sl_pct):
    """
    Main backtesting function that processes all tickers.

    Parameters
    ----------
    data : pd.DataFrame
        Input data with OHLC, signals, and indicators
    initial_capital : float
        Starting capital per trade
    fee_rate : float
        Transaction fee rate
    slippage_rate : float
        Slippage rate for entries
    entry_sl_pct : float
        Entry stop-loss percentage
    """
    data = data.copy()
    data['date'] = pd.to_datetime(data['date'])

    tickers = data['ticker'].unique()
    ticker_map = {
        ticker: data[data['ticker'] == ticker][["date", "open", "high", "low", "close", "Signal", "ATR_max"]].to_numpy()
        for ticker in tickers
    }

    all_trades = []
    for ticker in tqdm(tickers, desc="Backtesting"):
        trades = py_backtest_trades(
            ticker_map[ticker],
            ticker,
            initial_capital,
            fee_rate,
            slippage_rate,
            entry_sl_pct
        )
        all_trades.extend(trades)

    return pd.DataFrame(all_trades, columns=["Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price",
                                           "PnL", "Exit Reason", "Cumulative PnL", "Direction"])

# Usage
# Assuming 'data' is your input DataFrame with columns: Date, Ticker, Open, High, Low, Close
data = prepare_data(data)
tradebook = backtest_trades(
    data,
    initial_capital=100000,
    fee_rate=0.00025,
    slippage_rate=0.0005,
    entry_sl_pct=0.02
)
tradebook.to_csv("/content/drive/MyDrive/tradebook_Crypto_2025.csv", index=False)
tradebook

In [ ]:
data

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

class ATRIndicator:
    def __init__(self, high_col='high', low_col='low', close_col='close'):
        self.high_col = high_col
        self.low_col = low_col
        self.close_col = close_col

    def compute_atr(self, df, period=14, shift=1):
        high, low, close = df[self.high_col], df[self.low_col], df[self.close_col]
        tr = pd.concat([
            high - low,
            (high - close.shift(1)).abs(),
            (low - close.shift(1)).abs()
        ], axis=1).max(axis=1)
        atr = tr.rolling(window=period, min_periods=1).mean()
        return atr.shift(shift)

def compute_jma_and_signals(df, smooth_length=14, normalization_period=72,
                            upper_threshold=1.75, lower_threshold=-0.75,
                            rs_period=55, rs_percentile=70):
    df = df.copy()
    price = df['close']

    # JMA calculation
    alpha = 2.0 / (smooth_length + 1)
    jma_values = [price.iloc[0]]
    for p in price.iloc[1:]:
        jma_values.append(jma_values[-1] + (p - jma_values[-1]) * alpha)
    df['JMA'] = jma_values

    # Normalization using shifted JMA to avoid look-ahead
    df['JMA_shifted'] = df['JMA'].shift(1)
    df['Normalized_JMA'] = (df['JMA_shifted'] - df['JMA_shifted'].rolling(normalization_period).mean()) / \
                          df['JMA_shifted'].rolling(normalization_period).std()

    # RS Volatility using previous bar's OHLC
    o = df['open'].shift(1).replace(0, np.nan)
    h = df['high'].shift(1)
    l = df['low'].shift(1)
    c = df['close'].shift(1).replace(0, np.nan)

    log_hc = np.log(h / c)
    log_ho = np.log(h / o)
    log_lc = np.log(l / c)
    log_lo = np.log(l / o)

    df['RS_Volatility'] = ((log_hc * log_ho + log_lc * log_lo).rolling(window=rs_period).mean()) * 1e6
    df['RS_Threshold'] = df['RS_Volatility'].rolling(normalization_period).quantile(rs_percentile / 100.0)

    # Signals based on shifted/lagged data
    df['Signal'] = 0
    df.loc[(df['Normalized_JMA'] > upper_threshold) & (df['RS_Volatility'] > df['RS_Threshold']), 'Signal'] = 1
    df.loc[(df['Normalized_JMA'] < lower_threshold) & (df['RS_Volatility'] > df['RS_Threshold']), 'Signal'] = 2

    return df

def compute_atr_max(group, multiplier=0.1):
    return pd.concat([
        group['ATR_14'].rolling(20).max() * multiplier,
        group['ATR_5'].rolling(20).max() * multiplier
    ], axis=1).max(axis=1)

def prepare_data_with_configs(data, jma_configs):
    """
    Prepare data with different JMA configurations for each ticker
    """
    data = data.copy()
    data['Date'] = pd.to_datetime(data['Date'])

    # Process each ticker with its specific JMA configuration
    jma_results = []
    for ticker, group in data.groupby('Ticker'):
        group = group.sort_values('Date').reset_index(drop=True)

        if ticker in jma_configs:
            config = jma_configs[ticker]
            print(f"Processing {ticker} with config: {config}")
            group_result = compute_jma_and_signals(
                group,
                smooth_length=config.get('smooth_length', 14),
                normalization_period=config.get('normalization_period', 72),
                upper_threshold=config.get('upper_threshold', 1.75),
                lower_threshold=config.get('lower_threshold', -0.75),
                rs_period=config.get('rs_period', 55),
                rs_percentile=config.get('rs_percentile', 70)
            )
        else:
            print(f"Warning: No config found for {ticker}, using default parameters")
            group_result = compute_jma_and_signals(group)

        jma_results.append(group_result)

    data = pd.concat(jma_results, ignore_index=True)

    # Calculate ATR indicators
    atr_calc = ATRIndicator()
    data = data.sort_values(['Ticker', 'Date']).reset_index(drop=True)

    for period in [14, 5]:
        col_name = f'ATR_{period}'
        atr_results = []
        for ticker, group in data.groupby('Ticker'):
            atr_results.append(atr_calc.compute_atr(group, period=period, shift=1))
        data[col_name] = pd.concat(atr_results, ignore_index=True)

    # Calculate ATR max
    atr_max_results = []
    for ticker, group in data.groupby('Ticker'):
        atr_max_results.append(compute_atr_max(group))
    data['ATR_max'] = pd.concat(atr_max_results, ignore_index=True)

    # Shift signals to avoid look-ahead bias
    data['Signal'] = data.groupby('Ticker', group_keys=False)['Signal'].shift(1)
    data = data.dropna(subset=['Signal'])
    data['Signal'] = data['Signal'].astype(np.int8)

    return data

def py_backtest_trades(
    data: np.ndarray,
    ticker: str,
    capital: float,
    fee_rate: float,
    slippage_rate: float,
    entry_sl_pct: float,
):
    """
    Refined Backtest Engine with Your Exact Logic

    Logic:
    - Entry Candle: Only entry SL check
    - Candle 2+:
      1. First check trailing SL
      2. Then check direction (open & close in our favor)
      3. Exit at close price (with slippage) if direction unfavorable
    """

    def calc_pnl(entry_price, exit_price, qty, direction):
        """Helper to compute PnL with fees."""
        pnl = (exit_price - entry_price) * qty * direction
        fee = fee_rate * (entry_price + exit_price) * qty
        return pnl - fee

    n = data.shape[0]
    trades, cumulative_pnl = [], 0.0

    # --- Trade State ---
    position_open = False
    entry_price = sl_price = quantity = 0.0
    entry_time = last_exit_time = None
    direction = 0  # 1 = long, -1 = short

    # --- Previous Candle (for trailing SL) ---
    prev_high = prev_low = prev_close = prev_atr_max = 0.0

    for i in range(n):
        time, open_, high, low, close, signal, atr_max = (
            pd.to_datetime(data[i, 0]),
            float(data[i, 1]), float(data[i, 2]),
            float(data[i, 3]), float(data[i, 4]), int(data[i, 5]),
            float(data[i, 6])
        )

        # ======================
        #  ENTRY LOGIC
        # ======================
        if (
            not position_open
            and time != last_exit_time
            and signal in (1, 2)
        ):
            direction = 1 if signal == 1 else -1
            entry_price = open_ * (1 + slippage_rate if direction == 1 else 1 - slippage_rate)
            quantity = capital / entry_price
            entry_time, position_open = time, True

            # Flash-crash protection (constant entry SL)
            sl_price = entry_price * (1 - entry_sl_pct if direction == 1 else 1 + entry_sl_pct)

            # ENTRY CANDLE: Only check entry SL
            if (direction == 1 and low <= sl_price) or (direction == -1 and high >= sl_price):
                exit_price = sl_price
                net_pnl = calc_pnl(entry_price, exit_price, quantity, direction)
                cumulative_pnl += net_pnl

                trades.append((
                    ticker, entry_time, time, entry_price, exit_price,
                    net_pnl, "Entry SL", cumulative_pnl,
                    "long" if direction == 1 else "short"
                ))

                position_open, last_exit_time = False, time

            # Store current candle for next iteration
            prev_high, prev_low, prev_close, prev_atr_max = high, low, close, atr_max
            continue

        # ======================
        #  POSITION MANAGEMENT (CANDLE 2+)
        # ======================
        if position_open and time != entry_time:
            exit_triggered = False
            exit_price = None
            exit_reason = None

            # STEP 1: Check Trailing SL FIRST (Priority)
            # Update trailing SL using previous candle
            if direction == 1:
                sl_price = max(sl_price, prev_close - prev_atr_max)
            else:
                sl_price = min(sl_price, prev_close + prev_atr_max)

            # Check if trailing SL hit
            if (direction == 1 and low <= sl_price) or (direction == -1 and high >= sl_price):
                exit_price = sl_price
                exit_reason = "Trailing SL"
                exit_triggered = True

            # STEP 2: Check Direction (Open & Close) - Only if Trailing SL not hit
            else:
                # Check if BOTH open and close are unfavorable
                unfavorable_open = (direction == 1 and open_ <= entry_price) or (direction == -1 and open_ >= entry_price)
                unfavorable_close = (direction == 1 and close <= entry_price) or (direction == -1 and close >= entry_price)

                if unfavorable_open or unfavorable_close:
                    # Exit at CLOSE price with slippage
                    exit_price = close * (1 - slippage_rate if direction == 1 else 1 + slippage_rate)
                    exit_reason = "Not in direction (open/close)"
                    exit_triggered = True

            # Execute exit if any condition triggered
            if exit_triggered:
                net_pnl = calc_pnl(entry_price, exit_price, quantity, direction)
                cumulative_pnl += net_pnl

                trades.append((
                    ticker, entry_time, time, entry_price, exit_price,
                    net_pnl, exit_reason, cumulative_pnl,
                    "long" if direction == 1 else "short"
                ))

                position_open, last_exit_time = False, time

        # Store previous candle (for trailing SL reference)
        prev_high, prev_low, prev_close, prev_atr_max = high, low, close, atr_max

    return trades

def backtest_trades(data, entry_sl_pct, initial_capital, fee_rate, slippage_rate):
    data = data.copy()
    data['Date'] = pd.to_datetime(data['Date'])

    tickers = data['Ticker'].unique()
    ticker_map = {}

    for ticker in tickers:
        ticker_data = data[data['Ticker'] == ticker].sort_values('Date')
        ticker_map[ticker] = ticker_data[["Date", "open", "high", "low", "close", "Signal", "ATR_max"]].to_numpy(dtype=object)

    all_trades = []
    for ticker in tqdm(tickers, desc="Backtesting"):
        trades = py_backtest_trades(ticker_map[ticker], ticker, initial_capital, fee_rate, slippage_rate, entry_sl_pct)
        all_trades.extend(trades)

    return pd.DataFrame(all_trades, columns=["Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price",
                                           "PnL", "Exit Reason", "Cumulative PnL", "Direction"])

def run_multi_ticker_backtest(data):
    """Run backtest with different JMA configurations for each ticker"""

    # Define different JMA configurations for each ticker
    jma_configs = {
        'BTCUSDT': {
            'smooth_length': 14,
            'normalization_period': 72,
            'upper_threshold': 1.75,
            'lower_threshold': -0.75,
            'rs_period': 55,
            'rs_percentile': 70
        },
        'ETHUSDT': {
            'smooth_length': 14,
            'normalization_period': 60,
            'upper_threshold': 1.50,
            'lower_threshold': -1.00,
            'rs_period': 14,
            'rs_percentile': 10
        },
        'BNBUSDT': {
            'smooth_length': 14,
            'normalization_period': 108,
            'upper_threshold': 2.25,
            'lower_threshold': -1.25,
            'rs_period': 3,
            'rs_percentile': 10
        },
        'XRPUSDT': {
            'smooth_length': 14,
            'normalization_period': 72,
            'upper_threshold': 2.25,
            'lower_threshold': -1.75,
            'rs_period': 3,
            'rs_percentile': 10
        },
        'SOLUSDT': {
            'smooth_length': 14,
            'normalization_period': 72,
            'upper_threshold': 2.25,
            'lower_threshold': -1.75,
            'rs_period': 3,
            'rs_percentile': 10
        },
    }

    # Get actual ticker names from data
    actual_tickers = data['Ticker'].unique()
    print(f"Tickers in data: {actual_tickers}")

    # Prepare data with ticker-specific configurations
    prepared_data = prepare_data_with_configs(data, jma_configs)

    # Run backtest
    tradebook = backtest_trades(
        prepared_data,
        entry_sl_pct=0.01,
        initial_capital=100000,
        fee_rate=0.00025,
        slippage_rate=0.0005
    )

    # Print summary by ticker
    print("\n=== Backtest Summary by Ticker ===")
    for ticker in actual_tickers:
        ticker_trades = tradebook[tradebook['Ticker'] == ticker]
        if len(ticker_trades) > 0:
            total_pnl = ticker_trades['PnL'].sum()
            win_rate = (ticker_trades['PnL'] > 0).mean() * 100
            total_trades = len(ticker_trades)
            config = jma_configs.get(ticker, {})

            print(f"\n{ticker}:")
            print(f"  Config: smooth_length={config.get('smooth_length', 'N/A')}, "
                  f"thresholds=[{config.get('upper_threshold', 'N/A')}, {config.get('lower_threshold', 'N/A')}]")
            print(f"  Total Trades: {total_trades}")
            print(f"  Total PnL: ${total_pnl:.2f}")
            print(f"  Win Rate: {win_rate:.1f}%")

    return tradebook, prepared_data

def show_tickers_and_create_template(data):
    """Helper function to show available tickers and create a configuration template"""
    tickers = data['Ticker'].unique()
    print("Available tickers:", tickers)

    # Create a template configuration dictionary
    template_config = {}
    for i, ticker in enumerate(tickers):
        template_config[ticker] = {
            'smooth_length': 14 + i * 3,
            'normalization_period': 72 - i * 5,
            'upper_threshold': 1.75 + i * 0.25,
            'lower_threshold': -0.75 - i * 0.25,
            'rs_period': 55 + i * 5,
            'rs_percentile': 70 + i * 2
        }

    print("\nTemplate configuration:")
    for ticker, config in template_config.items():
        print(f"'{ticker}': {config},")

    return template_config

# Usage example:
if __name__ == "__main__":
    # Load your data
    # data = pd.read_csv('your_data.csv')

    # Run backtest
    tradebook, prepared_data = run_multi_ticker_backtest(data)

    # Save results
    tradebook.to_csv("/content/drive/MyDrive/tradebook_Crypto_2020_2025.csv", index=False)

In [ ]:
tradebook